# 跨資料集橋接分析 — Topic Taxonomy

本筆記本展示如何透過 **共同主題分類法 (Topic Taxonomy)** 將 arXiv 論文和 GitHub 熱門專案連結起來。

## 方法論

我們定義了約 10 個 AI/ML 主題（如 NLP、Computer Vision、Machine Learning 等），
並建立兩套對應規則：

1. **arXiv → Topic**：根據 `primary_category`（如 cs.CL → NLP, cs.CV → Computer Vision）
2. **GitHub → Topic**：根據 repo 名稱和 owner 的關鍵字匹配（如 llm, gpt → NLP）

這使得兩個原本獨立的資料集可以在 Tableau 中用相同的「主題」維度進行比較。

---

## 環境設定

In [ ]:
import pandas as pd

from ai_ecosystem.bridge.taxonomy import (
    classify_arxiv_topic,
    classify_github_topic,
    get_taxonomy,
)
from ai_ecosystem.bridge.matcher import (
    bridge_arxiv_topics,
    bridge_github_topics,
    bridge_combined_timeline,
    bridge_topic_summary,
)
from ai_ecosystem.ingest.arxiv import load_arxiv_data
from ai_ecosystem.ingest.github_trending import load_github_trending

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

---

## 1. 主題分類法 (Topic Taxonomy)

我們定義的 taxonomy 包含每個主題對應的 arXiv 類別和 GitHub 關鍵字。
這張表本身也會匯出為 CSV，方便在 Tableau 中作為參考維度表使用。

In [ ]:
taxonomy = get_taxonomy()
print(f"Taxonomy 共 {len(taxonomy)} 筆對應規則，涵蓋 {taxonomy['topic'].nunique()} 個主題")
taxonomy

---

## 2. arXiv 論文按主題分類

將每篇論文的 `primary_category` 對應到我們定義的主題，
然後按主題和月份彙總論文數量。

In [ ]:
arxiv_df = load_arxiv_data()
arxiv_by_topic = bridge_arxiv_topics(arxiv_df)
print(f"arXiv 按主題分類：{len(arxiv_by_topic)} 列")
arxiv_by_topic

### arXiv 各主題的論文總數

In [ ]:
arxiv_by_topic.groupby("topic")["paper_count"].sum().sort_values(ascending=False)

---

## 3. GitHub Repos 按主題分類

透過關鍵字比對將每個 repo 歸類到主題。
注意：一個 repo 可能同時匹配多個主題（例如 pytorch-yolo 會同時歸入 Deep Learning Frameworks 和 Computer Vision）。

> **限制**：僅根據 repo 名稱和 owner 名稱進行比對，無法捕捉 repo description 中的關鍵字。

In [ ]:
github_df = load_github_trending()
github_by_topic = bridge_github_topics(github_df)
print(f"GitHub 按主題分類：{len(github_by_topic)} 列")
github_by_topic.head(15)

### GitHub 各主題的 Repo 總數

In [ ]:
github_by_topic.groupby("topic")["repo_count"].sum().sort_values(ascending=False)

---

## 4. 跨資料集時間軸

將 arXiv 和 GitHub 的主題彙總合併到同一張表，讓 Tableau 可以用雙軸圖呈現。

> **注意**：arXiv 僅有 2025-09 ~ 2026-04 的資料，GitHub 涵蓋 2013-08 ~ 2025-11。
> 重疊期間僅約 3 個月 (2025-09 ~ 2025-11)，因此跨資料集的時間比較需謹慎解讀。

In [ ]:
timeline = bridge_combined_timeline(arxiv_by_topic, github_by_topic)
print(f"合併時間軸：{len(timeline)} 列，時間跨度 {timeline['month'].min()} ~ {timeline['month'].max()}")
timeline.head(15)

---

## 5. 主題總覽

每個主題在兩個資料集中各有多少資料？這張表是 Tableau Dashboard 的核心參考。

In [ ]:
summary = bridge_topic_summary(arxiv_by_topic, github_by_topic)
summary

---

## 6. 匯出所有 CSV

執行以下程式碼，將描述性 EDA (12 張表) + 橋接分析 (5 張表) 全部匯出至 `data/processed/`。

In [ ]:
from ai_ecosystem.analysis.export_tableau import export_all

exported = export_all()
print(f"\n已匯出 {len(exported)} 個 CSV 檔案：")
for p in exported:
    print(f"  {p}")

---

## 方法限制

| 項目 | 說明 |
|------|------|
| 關鍵字比對 | 僅匹配 repo 名稱和 owner，未考慮 repo description 或 README 內容 |
| 時間重疊 | arXiv (2025-09~2026-04) 和 GitHub (2013-08~2025-11) 僅有約 3 個月重疊 |
| 主題粒度 | 手工定義 ~10 個主題，可能無法捕捉所有子領域 |
| 多主題 repo | 一個 repo 可歸入多個主題，會造成 repo_count 的重複計算 |

這些限制在 side project 的範圍內是可接受的。如需更精確的分類，
可考慮使用 BERTopic 等 NLP 方法（見 `bridge/topic_model.py` placeholder）。